# Reasoning Models — Expert

> **Section 01 — Topic 08 — expert**

**Prerequisites:** `08-reasoning-models/foundations.ipynb`, `08-reasoning-models/advanced.ipynb`

**What you'll learn:**
- Process reward models: outcome vs process supervision
- MCTS adapted for language model reasoning
- Verification and self-correction pipelines
- How reasoning models are trained (STaR, RL from outcome supervision)
- Fundamental limits of LLM reasoning

**What you'll build:**
- A process reward model scorer
- MCTS for reasoning over language
- A verify-and-retry pipeline

## Setup

In [ ]:
!pip install -q openai anthropic torch transformers matplotlib numpy

In [ ]:
import os
import re
import json
import time
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-key-here"))
MODEL = "gpt-4o-mini"

def ask(prompt, temperature=0.0, max_tokens=1024):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

print(f"Using model: {MODEL}")

---
## 1. Process Reward Models — Outcome vs Process Supervision

### The problem with outcome-based evaluation

Standard evaluation gives a binary signal: was the final answer right or wrong? This is **outcome reward**. But a model might get the right answer through wrong reasoning, or wrong answer through mostly-correct reasoning.

**Process Reward Models (PRMs)** evaluate each *step* of the reasoning, not just the final answer. This gives much denser supervision and better identifies where reasoning goes wrong.

Lightman et al. (2023) showed that process supervision dramatically outperforms outcome supervision for training math reasoning models.

### Types of supervision:

| Type | What's scored | Signal density | Human cost |
|------|--------------|----------------|------------|
| **Outcome (ORM)** | Final answer only | Very sparse | Low |
| **Process (PRM)** | Each reasoning step | Dense | High |
| **Automatic process** | Each step, auto-labeled | Dense | Low |

In [ ]:
class ProcessRewardModel:
    """
    A simulated Process Reward Model that scores each step of reasoning.
    
    In production, this would be a trained classifier. Here we use an LLM
    as the evaluator (a common bootstrapping approach).
    """
    
    def __init__(self, model=MODEL):
        self.model = model
    
    def score_step(self, problem, steps_so_far, current_step):
        """
        Score a single reasoning step on a scale of -1 to 1.
        -1 = definitely wrong
         0 = neutral/unclear
         1 = definitely correct
        """
        prompt = f"""You are evaluating a reasoning step in a math solution.

Problem: {problem}

Previous steps:
{steps_so_far if steps_so_far else '(none)'}

Current step to evaluate:
{current_step}

Is this step mathematically correct AND does it make progress toward solving the problem?

Score this step:
- "+1" if the step is correct and makes useful progress
- "0" if the step is correct but not particularly useful
- "-1" if the step contains an error

Reply with just the score (+1, 0, or -1) and a brief explanation."""
        
        response = ask(prompt, temperature=0.0)
        
        # Parse score
        if "+1" in response[:10] or "1" == response.strip()[0]:
            score = 1.0
        elif "-1" in response[:10]:
            score = -1.0
        else:
            score = 0.0
        
        return score, response
    
    def score_full_solution(self, problem, solution_text):
        """
        Score every step in a solution.
        Returns list of (step, score, explanation).
        """
        # Split solution into steps
        steps = [s.strip() for s in solution_text.split('\n') if s.strip()]
        
        results = []
        steps_so_far = ""
        
        for step in steps:
            score, explanation = self.score_step(problem, steps_so_far, step)
            results.append({
                'step': step,
                'score': score,
                'explanation': explanation,
            })
            steps_so_far += f"\n{step}"
        
        return results

# Test PRM on a solution with a deliberate error
prm = ProcessRewardModel()

problem = "What is the area of a triangle with base 8 and height 6?"

# Solution with an error in step 2
wrong_solution = """Step 1: The formula for the area of a triangle is A = (1/2) * base * height.
Step 2: Plugging in: A = (1/2) * 8 * 6 = 8 * 6 / 2 = 48 / 2 = 24.
Step 3: Wait, let me recalculate. A = (1/2) * 8 * 6 = 4 * 6 = 24.
Step 4: The area is 24 square units."""

print(f"Problem: {problem}")
print(f"\nScoring solution steps:")
scores = prm.score_full_solution(problem, wrong_solution)

for r in scores:
    emoji = {1.0: 'GOOD', 0.0: 'NEUTRAL', -1.0: 'ERROR'}[r['score']]
    print(f"\n  [{emoji}] {r['step'][:80]}")
    print(f"        Score: {r['score']}, Reason: {r['explanation'][:100]}")

In [ ]:
# Visualize step-by-step scores
fig, ax = plt.subplots(figsize=(12, 4))

step_scores = [r['score'] for r in scores]
step_labels = [f"Step {i+1}" for i in range(len(scores))]
colors = ['#2ecc71' if s > 0 else '#e74c3c' if s < 0 else '#f39c12' for s in step_scores]

bars = ax.bar(step_labels, step_scores, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Process Reward Score')
ax.set_title('Process Reward Model — Step-by-Step Evaluation')
ax.set_ylim(-1.3, 1.3)
ax.grid(axis='y', alpha=0.3)

# Add step text annotations
for bar, r in zip(bars, scores):
    text = r['step'][:40] + '...' if len(r['step']) > 40 else r['step']
    ax.text(bar.get_x() + bar.get_width()/2, -1.2, text,
            ha='center', fontsize=7, rotation=0, style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# PRM-guided best-of-N selection
def prm_best_of_n(problem, n=5, expected_answer=None):
    """
    Generate N solutions and use PRM to pick the best one.
    This is how PRMs are used at inference time.
    """
    solutions = []
    
    for i in range(n):
        response = ask(
            f"Solve step by step:\n{problem}",
            temperature=0.7
        )
        
        # Score each step
        step_scores = prm.score_full_solution(problem, response)
        
        # Aggregate: minimum step score (weakest link)
        min_score = min(r['score'] for r in step_scores) if step_scores else -1
        avg_score = np.mean([r['score'] for r in step_scores]) if step_scores else -1
        
        solutions.append({
            'text': response,
            'min_score': min_score,
            'avg_score': avg_score,
            'step_scores': step_scores,
        })
        print(f"  Solution {i+1}: min_score={min_score:.1f}, avg_score={avg_score:.2f}")
    
    # Select best by average score
    best = max(solutions, key=lambda s: s['avg_score'])
    return best, solutions

problem = "If a car travels at 45 mph for 2 hours and then 60 mph for 1.5 hours, what is the average speed for the entire trip?"
print(f"Problem: {problem}\n")

best, all_sols = prm_best_of_n(problem, n=3)
print(f"\nBest solution (avg_score={best['avg_score']:.2f}):")
print(best['text'][:300])

---
## 2. MCTS for Reasoning

Monte Carlo Tree Search (MCTS) was the key algorithm behind AlphaGo. The insight: we can adapt it for **language model reasoning** by treating reasoning steps as "moves" in a game.

### MCTS phases:
1. **Selection:** Navigate the tree using UCB1 to balance exploration/exploitation
2. **Expansion:** Add a new reasoning step (child node)
3. **Simulation:** Roll out the reasoning to completion
4. **Backpropagation:** Update node statistics based on whether the rollout succeeded

$$\text{UCB1}(s) = \bar{Q}(s) + c \cdot \sqrt{\frac{\ln N(\text{parent})}{N(s)}}$$

where $\bar{Q}(s)$ is the average reward and $N(s)$ is the visit count.

In [ ]:
class MCTSNode:
    """A node in the MCTS reasoning tree."""
    
    def __init__(self, state, parent=None):
        self.state = state            # Partial reasoning text
        self.parent = parent
        self.children = []
        self.visits = 0
        self.total_reward = 0.0
    
    @property
    def avg_reward(self):
        return self.total_reward / self.visits if self.visits > 0 else 0
    
    def ucb1(self, exploration_weight=1.41):
        if self.visits == 0:
            return float('inf')  # Unexplored nodes get priority
        exploitation = self.avg_reward
        exploration = exploration_weight * math.sqrt(
            math.log(self.parent.visits) / self.visits
        )
        return exploitation + exploration
    
    def best_child(self):
        return max(self.children, key=lambda c: c.ucb1())


class MCTSReasoner:
    """
    Monte Carlo Tree Search for reasoning.
    Treats reasoning steps as moves in a search tree.
    """
    
    def __init__(self, problem, expected_answer=None, max_depth=4, n_simulations=20):
        self.problem = problem
        self.expected_answer = expected_answer
        self.max_depth = max_depth
        self.n_simulations = n_simulations
        self.root = MCTSNode(state="")
        self.stats = {'nodes_created': 0, 'rollouts': 0}
    
    def expand(self, node):
        """Generate possible next reasoning steps."""
        prompt = f"""Problem: {self.problem}

Reasoning so far:
{node.state if node.state else '(start)'}

Generate 3 different possible next reasoning steps. Be specific and mathematical.
Format: one per line, numbered 1-3."""
        
        response = ask(prompt, temperature=0.8, max_tokens=300)
        
        steps = []
        for line in response.strip().split('\n'):
            cleaned = re.sub(r'^\d+[\.\)\:]\s*', '', line.strip())
            if cleaned and len(cleaned) > 5:
                steps.append(cleaned)
        
        for step in steps[:3]:
            new_state = f"{node.state}\n{step}" if node.state else step
            child = MCTSNode(state=new_state, parent=node)
            node.children.append(child)
            self.stats['nodes_created'] += 1
        
        return node.children
    
    def simulate(self, node):
        """Roll out reasoning to completion and evaluate."""
        prompt = f"""Problem: {self.problem}

Reasoning so far:
{node.state}

Complete this reasoning and give a final numerical answer."""
        
        response = ask(prompt, temperature=0.5, max_tokens=500)
        self.stats['rollouts'] += 1
        
        # Evaluate: did we get the right answer?
        if self.expected_answer:
            numbers = re.findall(r'[-+]?\d*\.?\d+', response)
            if numbers and numbers[-1] == self.expected_answer:
                return 1.0  # Correct
            return 0.0  # Incorrect
        else:
            # Without ground truth, use LLM confidence as proxy
            eval_prompt = f"""Rate the quality of this solution (1-10):\n{response}\nScore:"""
            score_text = ask(eval_prompt, max_tokens=10)
            nums = re.findall(r'\d+', score_text)
            return int(nums[0]) / 10.0 if nums else 0.5
    
    def backpropagate(self, node, reward):
        """Update statistics from node to root."""
        while node is not None:
            node.visits += 1
            node.total_reward += reward
            node = node.parent
    
    def search(self):
        """Run MCTS for n_simulations iterations."""
        for sim in range(self.n_simulations):
            # Selection: traverse tree using UCB1
            node = self.root
            depth = 0
            
            while node.children and depth < self.max_depth:
                node = node.best_child()
                depth += 1
            
            # Expansion: if not at max depth, expand
            if depth < self.max_depth and node.visits > 0:
                children = self.expand(node)
                if children:
                    node = children[0]  # Pick first unexplored child
            
            # Simulation: roll out to completion
            reward = self.simulate(node)
            
            # Backpropagation
            self.backpropagate(node, reward)
            
            if (sim + 1) % 5 == 0:
                print(f"  Simulation {sim+1}/{self.n_simulations}: "
                      f"best avg reward = {self.get_best_path_reward():.2f}")
        
        return self.get_best_path()
    
    def get_best_path_reward(self):
        node = self.root
        while node.children:
            node = max(node.children, key=lambda c: c.avg_reward)
        return node.avg_reward
    
    def get_best_path(self):
        """Get the best reasoning path by following highest-reward children."""
        path = []
        node = self.root
        while node.children:
            node = max(node.children, key=lambda c: c.avg_reward)
            path.append(node)
        return path

print("MCTS Reasoner ready")

In [ ]:
# Run MCTS on a math problem
problem = "What is 37 * 28 + 156 - 89?"
expected = "1103"  # 1036 + 156 - 89 = 1103

print(f"Problem: {problem}")
print(f"Expected: {expected}")
print()

mcts = MCTSReasoner(problem, expected_answer=expected, max_depth=3, n_simulations=15)
best_path = mcts.search()

print(f"\nMCTS Statistics:")
print(f"  Nodes created: {mcts.stats['nodes_created']}")
print(f"  Rollouts: {mcts.stats['rollouts']}")

print(f"\nBest reasoning path:")
for i, node in enumerate(best_path):
    print(f"  Step {i+1} (reward={node.avg_reward:.2f}, visits={node.visits}):")
    # Show just the last line of the state (the new step)
    lines = node.state.strip().split('\n')
    print(f"    {lines[-1][:100]}")

---
## 3. Verification and Self-Correction

A critical capability: can models **verify** their own answers and **correct** mistakes? Research shows this is hard — models often fail to catch their own errors. But we can build pipelines that improve reliability.

### The verify-then-retry pattern:
1. Generate a solution
2. Independently verify the solution
3. If verification fails, generate a new solution informed by the error
4. Repeat until verified or max retries

In [ ]:
def verify_solution(problem, solution):
    """Independent verification of a solution."""
    prompt = f"""You are a careful math verifier. Check this solution for errors.

Problem: {problem}

Proposed solution:
{solution}

Verify EACH step. If you find an error, explain exactly what's wrong.
At the end, state: VERIFIED (if correct) or ERROR: [description] (if wrong)."""
    
    verification = ask(prompt, temperature=0.0)
    
    is_correct = "VERIFIED" in verification and "ERROR" not in verification.split("VERIFIED")[0]
    return is_correct, verification


def verify_and_retry(problem, max_retries=3):
    """
    Generate, verify, and retry pipeline.
    """
    attempts = []
    feedback = ""
    
    for attempt in range(max_retries):
        # Generate solution (incorporate feedback from previous attempts)
        if feedback:
            prompt = f"""Solve this problem step by step.

Problem: {problem}

Previous attempt had this error: {feedback}
Please solve it correctly this time."""
        else:
            prompt = f"Solve this problem step by step.\n\nProblem: {problem}"
        
        solution = ask(prompt, temperature=0.2)
        
        # Verify
        is_correct, verification = verify_solution(problem, solution)
        
        attempts.append({
            'attempt': attempt + 1,
            'solution': solution,
            'verification': verification,
            'passed': is_correct,
        })
        
        print(f"  Attempt {attempt + 1}: {'PASSED' if is_correct else 'FAILED'}")
        
        if is_correct:
            return solution, attempts
        
        # Extract error feedback for next attempt
        error_match = re.search(r'ERROR:\s*(.+)', verification)
        feedback = error_match.group(1) if error_match else verification[-200:]
    
    # Return last attempt even if unverified
    return solution, attempts

# Test verify-and-retry
problem = (
    "A store has a 'buy 2 get 1 free' promotion on items that cost $30 each. "
    "You also have a 15% off coupon for the entire purchase (applied after the promotion). "
    "How much do you pay for 7 items?"
)

print(f"Problem: {problem}\n")
solution, attempts = verify_and_retry(problem)
print(f"\nFinal solution:")
print(solution[:300])

In [ ]:
# Visualize retry success rates
# Run multiple problems through verify-and-retry
test_problems = [
    "What is the compound interest on $5000 at 8% per year for 3 years?",
    "A train leaves at 9:00 AM going 60 mph. Another leaves at 10:00 AM going 80 mph. When do they meet if they're 300 miles apart?",
    "What is the probability of getting exactly 3 heads in 5 fair coin flips?",
    "How many diagonals does a regular octagon have?",
    "If f(x) = 2x^2 + 3x - 5, what is f(4)?",
]

retry_stats = {1: 0, 2: 0, 3: 0, 'failed': 0}

for i, prob in enumerate(test_problems):
    print(f"\nProblem {i+1}: {prob[:60]}...")
    _, attempts = verify_and_retry(prob, max_retries=3)
    
    passed_at = None
    for a in attempts:
        if a['passed']:
            passed_at = a['attempt']
            break
    
    if passed_at:
        retry_stats[passed_at] += 1
    else:
        retry_stats['failed'] += 1

print("\n" + "=" * 40)
print("Retry statistics:")
for k, v in retry_stats.items():
    print(f"  {'Attempt ' + str(k) if isinstance(k, int) else 'Failed'}: {v}/{len(test_problems)}")

---
## 4. Training Reasoning Models

How do models like o1 learn to reason? The key techniques:

### 4.1 STaR — Self-Taught Reasoner

Zelikman et al. (2022): Bootstrap reasoning ability from a small set of examples.

1. Given a problem, generate a reasoning chain
2. If the chain reaches the correct answer, keep it as training data
3. If wrong, provide the correct answer and ask the model to generate a rationalization
4. Fine-tune on the collected correct chains
5. Repeat

### 4.2 RL from Outcome Supervision

Train with reinforcement learning where the reward is simply whether the final answer is correct. The model learns which reasoning patterns lead to correct answers.

In [ ]:
# Simulate the STaR bootstrapping process

def star_iteration(problems, model_generate_fn):
    """
    One iteration of STaR:
    1. Generate reasoning chains
    2. Keep correct ones
    3. For incorrect ones, generate rationalizations
    4. Return training data
    """
    training_data = []
    
    for p in problems:
        # Generate reasoning chain
        response = model_generate_fn(p['question'])
        extracted = re.findall(r'[-+]?\d*\.?\d+', response)
        predicted = extracted[-1] if extracted else None
        
        if predicted == p['answer']:
            # Correct! Keep this chain
            training_data.append({
                'question': p['question'],
                'reasoning': response,
                'source': 'correct_generation',
            })
        else:
            # Wrong — generate rationalization with the correct answer
            rationalization_prompt = f"""The answer to this problem is {p['answer']}.
Show the step-by-step reasoning that leads to this answer.

Problem: {p['question']}

Step-by-step reasoning leading to {p['answer']}:"""
            
            rationalization = ask(rationalization_prompt, temperature=0.3)
            training_data.append({
                'question': p['question'],
                'reasoning': rationalization,
                'source': 'rationalization',
            })
    
    return training_data

# Simulate
problems = [
    {"question": "What is 23 * 17?", "answer": "391"},
    {"question": "If x^2 = 144, what is x?", "answer": "12"},
    {"question": "What is 15% of 280?", "answer": "42"},
    {"question": "Sum the first 8 positive integers.", "answer": "36"},
]

generate_fn = lambda q: ask(f"{q}\nThink step by step.", temperature=0.5)

print("STaR Iteration 1:")
training_data = star_iteration(problems, generate_fn)

correct_gen = sum(1 for d in training_data if d['source'] == 'correct_generation')
rationalized = sum(1 for d in training_data if d['source'] == 'rationalization')

print(f"  Correct generations: {correct_gen}/{len(problems)}")
print(f"  Rationalizations:   {rationalized}/{len(problems)}")
print(f"  Total training examples: {len(training_data)}")

print("\nIn a real STaR setup, we would now fine-tune the model on these examples")
print("and repeat, with each iteration improving reasoning quality.")

In [ ]:
# Visualize STaR iteration concept
fig, ax = plt.subplots(figsize=(10, 5))

# Simulated STaR performance over iterations (based on paper results)
iterations = list(range(1, 8))
accuracy_star = [0.45, 0.55, 0.62, 0.68, 0.72, 0.75, 0.77]
accuracy_baseline = [0.45] * 7  # No improvement without STaR
accuracy_ft_only = [0.45, 0.50, 0.52, 0.53, 0.53, 0.53, 0.53]  # SFT plateaus

ax.plot(iterations, accuracy_star, 'o-', color='#2ecc71', linewidth=2, label='STaR', markersize=8)
ax.plot(iterations, accuracy_ft_only, 's--', color='#f39c12', linewidth=2, label='SFT only', markersize=6)
ax.plot(iterations, accuracy_baseline, ':', color='#bdc3c7', linewidth=2, label='No training')

ax.set_xlabel('Training Iteration')
ax.set_ylabel('Accuracy')
ax.set_title('STaR: Self-Taught Reasoner — Bootstrapping Performance')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 0.9)

plt.tight_layout()
plt.show()

print("STaR key insight: rationalizations from wrong answers provide")
print("training signal that SFT on correct examples alone cannot.")

---
## 5. Limits of Reasoning — Formal vs Pattern Matching

A critical question: are LLMs actually *reasoning*, or just doing sophisticated pattern matching? Let's design tests that probe this distinction.

### Tests that distinguish reasoning from pattern matching:
1. **Novel formats:** Same problem in an unusual representation
2. **Adversarial examples:** Problems designed to trigger common errors
3. **Compositional generalization:** Combining known skills in new ways
4. **Counterfactual reasoning:** What if the rules were different?

In [ ]:
# Test 1: Novel formats
# Standard format vs unusual format of the same problem
standard_problems = [
    "What is 3 + 5?",
    "What is 7 * 8?",
    "What is 100 / 4?",
]

novel_format_problems = [
    "If # means addition, what is 3 # 5?",
    "In a system where @ represents multiplication, compute 7 @ 8.",
    "Using the operation DIVIDE(a,b) = a/b, what is DIVIDE(100, 4)?",
]

expected = ["8", "56", "25"]

print("Test: Novel Formats")
print("="*50)

standard_correct = 0
novel_correct = 0

for s, n, e in zip(standard_problems, novel_format_problems, expected):
    s_ans = ask(f"{s} Answer with just the number.")
    n_ans = ask(f"{n} Answer with just the number.")
    
    s_num = re.findall(r'\d+', s_ans)
    n_num = re.findall(r'\d+', n_ans)
    
    s_correct = s_num and s_num[0] == e
    n_correct = n_num and n_num[0] == e
    standard_correct += int(s_correct)
    novel_correct += int(n_correct)
    
    print(f"  Standard: {s:<30} → {s_ans.strip():<10} {'OK' if s_correct else 'FAIL'}")
    print(f"  Novel:    {n:<30} → {n_ans.strip():<10} {'OK' if n_correct else 'FAIL'}")
    print()

print(f"Standard: {standard_correct}/{len(standard_problems)}")
print(f"Novel:    {novel_correct}/{len(novel_format_problems)}")

In [ ]:
# Test 2: Counterfactual reasoning
# Can the model reason under different rules?

counterfactual_tests = [
    {
        "prompt": "In a world where 1+1=3, what is 2+2? (Hint: addition adds an extra 1 to the normal result)",
        "expected": "5",
    },
    {
        "prompt": "Suppose we define a new operation * where a * b = a + b + 1. What is 3 * 4?",
        "expected": "8",
    },
    {
        "prompt": "In base 5, what is 3 + 4? (Give the answer in base 5)",
        "expected": "12",  # 7 in decimal = 12 in base 5
    },
]

print("Test: Counterfactual Reasoning")
print("="*50)

for test in counterfactual_tests:
    response = ask(f"{test['prompt']}\nLet's think step by step. End with 'Answer: X'")
    nums = re.findall(r'Answer:\s*(\d+)', response)
    predicted = nums[-1] if nums else re.findall(r'\d+', response.split('\n')[-1])[-1] if re.findall(r'\d+', response.split('\n')[-1]) else "?"
    correct = predicted == test['expected']
    
    print(f"  Q: {test['prompt'][:70]}...")
    print(f"  Expected: {test['expected']}, Got: {predicted} {'OK' if correct else 'FAIL'}")
    print()

In [ ]:
# Test 3: Adversarial — problems designed to trigger common errors
adversarial = [
    {
        "prompt": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "wrong_answer": "0.10",  # The intuitive-but-wrong answer
        "correct_answer": "0.05",
    },
    {
        "prompt": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        "wrong_answer": "100",  # The intuitive-but-wrong answer
        "correct_answer": "5",
    },
    {
        "prompt": "In a lake, there is a patch of lily pads. Every day, the patch doubles in size. If it takes 48 days for the patch to cover the entire lake, how long would it take for the patch to cover half of the lake?",
        "wrong_answer": "24",  # The intuitive-but-wrong answer
        "correct_answer": "47",
    },
]

print("Test: Adversarial (Cognitive Reflection)")
print("="*50)

methods = {
    "Direct": lambda q: ask(f"{q}\nAnswer with just the number."),
    "CoT": lambda q: ask(f"{q}\nLet's think step by step."),
}

results_by_method = {}
for method_name, method_fn in methods.items():
    correct = 0
    fell_for_trap = 0
    
    for test in adversarial:
        response = method_fn(test['prompt'])
        nums = re.findall(r'\d+\.?\d*', response)
        predicted = nums[-1] if nums else "?"
        
        is_correct = predicted == test['correct_answer']
        is_trapped = predicted == test['wrong_answer']
        correct += int(is_correct)
        fell_for_trap += int(is_trapped)
    
    results_by_method[method_name] = {'correct': correct, 'trapped': fell_for_trap}
    print(f"  {method_name}: {correct}/{len(adversarial)} correct, {fell_for_trap}/{len(adversarial)} fell for trap")

# Visualize
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(methods))
width = 0.3

correct_vals = [r['correct'] for r in results_by_method.values()]
trapped_vals = [r['trapped'] for r in results_by_method.values()]

ax.bar(x - width/2, correct_vals, width, label='Correct', color='#2ecc71')
ax.bar(x + width/2, trapped_vals, width, label='Fell for trap', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(list(methods.keys()))
ax.set_ylabel('Count')
ax.set_title('Adversarial Problems: CoT Helps Avoid Traps')
ax.legend()
ax.set_ylim(0, len(adversarial) + 0.5)
plt.tight_layout()
plt.show()

### Fundamental limits

Current LLM reasoning has clear limitations:

1. **Compositional generalization:** Models struggle to combine learned skills in truly novel ways
2. **Consistent formal reasoning:** Multi-step logical deductions often accumulate errors
3. **Adversarial robustness:** Small changes to problem format can break reasoning
4. **Calibration:** Models are often overconfident in wrong reasoning chains
5. **Novel problem types:** Performance drops sharply on problem types not in training data

These limitations motivate the approaches we've studied: process supervision, search (ToT, MCTS), tool augmentation, and verification pipelines.

---
## Summary

| Topic | Key Insight |
|-------|------------|
| **Process Reward Models** | Evaluate each reasoning step, not just the final answer — catches errors earlier |
| **MCTS for Reasoning** | Treat reasoning as a search problem; explore, evaluate, and backpropagate |
| **Verify-and-Retry** | Independent verification + retry is more reliable than single-pass |
| **STaR** | Bootstrap reasoning ability by training on correct chains + rationalizations |
| **Limits** | LLMs do sophisticated pattern matching, not formal reasoning — design systems accordingly |

**Next:** `08-reasoning-models/build.ipynb` — build a complete reasoning pipeline with multi-strategy generation, verification, and scoring